# 08 — Demographic Consistency of Feature–FIQA Relationships

This notebook extends the project from overall feature effects to
**intersectional demographic consistency**.

It continues from:

- **Notebook 05**, which estimated univariate feature–FIQA associations;
- **Notebook 06**, which estimated adjusted feature effects;
- **Notebook 07**, which explained the final nonlinear prediction model.

## Research objective

> **Do facial and image characteristics show comparable relationships with
> CR-FIQA scores across the six DiveFace demographic groups?**

The analysis includes:

1. demographic CR-FIQA score distributions;
2. group-wise univariate effects;
3. one common adjusted regression specification across all groups;
4. group-specific adjusted coefficients;
5. formal feature × demographic-group interaction tests;
6. a separate facial-hair analysis restricted to male groups;
7. a concise paper-ready evidence table.

## Important interpretation rule

The main evidence for demographic heterogeneity is the **FDR-adjusted global
interaction test**. Differences in coefficient signs or magnitudes across
heatmap cells are descriptive and are not interpreted alone as proof of
different demographic effects.

## Feature handling

- `age` is intentionally excluded.
- continuous and graded features are standardized once on the pooled sample;
- binary variables enter the common adjusted model only when both states have
  sufficient support in every group;
- facial-hair features are evaluated separately in male groups because their
  support differs strongly by gender;
- standard errors are clustered by identity when `cls` is available, otherwise
  HC3 robust standard errors are used.

## Outputs

```text
results/08_demographic_consistency/
├── figures/
└── tables/
```

## 1. Shared project setup

In [ ]:
from pathlib import Path

setup_candidates = [
    Path.cwd() / "00_colab_setup.ipynb",
    Path.cwd() / "notebooks" / "00_colab_setup.ipynb",
    Path.cwd().parent / "notebooks" / "00_colab_setup.ipynb",
    Path("/content/drive/MyDrive/FIQA_Project/notebooks/00_colab_setup.ipynb"),
    Path("/content/drive/MyDrive/FIQA_Project/00_colab_setup.ipynb"),
]

SETUP_NOTEBOOK = next(
    (path for path in setup_candidates if path.exists()),
    None,
)

if SETUP_NOTEBOOK is None:
    checked_paths = "\n".join(f"- {path}" for path in setup_candidates)
    raise FileNotFoundError(
        "00_colab_setup.ipynb could not be found.\n"
        "Keep all notebooks in the same notebooks/ directory or update "
        "setup_candidates.\n\n"
        f"Checked:\n{checked_paths}"
    )

print(f"Running setup notebook: {SETUP_NOTEBOOK}")
get_ipython().run_line_magic("run", f'"{SETUP_NOTEBOOK}"')

## 2. Imports and analysis configuration

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor,
)

warnings.filterwarnings("ignore", category=UserWarning)

TARGET = "cr_fiqa_score"
GROUP_COLUMN = "group"
IDENTITY_COLUMN = "cls"

CONTINUOUS_FEATURES = [
    "smile",
    "head_roll",
    "head_yaw",
    "head_pitch",
    "blur",
    "exposure",
    "noise",
]

BINARY_FEATURES = [
    "mask",
    "headWear",
    "glasses",
    "eye_makeup",
    "lip_makeup",
    "forehead_occluded",
    "eye_occluded",
    "mouth_occluded",
]

FACIAL_HAIR_FEATURES = [
    "moustache",
    "beard",
    "sideburns",
]

EXPECTED_GROUP_ORDER = [
    "Asian Woman",
    "Asian Man",
    "Black Woman",
    "Black Man",
    "Caucasian Woman",
    "Caucasian Man",
]

MIN_STATE_COUNT = 15
MIN_NONZERO_HAIR = 15
MAX_VIF = 10.0
ALPHA = 0.05
FDR_METHOD = "fdr_bh"

## 3. Input and output paths

In [ ]:
DATA_FILE = PROJECT_PATH / "diveface_fiqa_merged.csv"

RESULTS_PATH = (
    PROJECT_PATH
    / "results"
    / "08_demographic_consistency"
)
TABLES_PATH = RESULTS_PATH / "tables"
FIGURES_PATH = RESULTS_PATH / "figures"

for path in [RESULTS_PATH, TABLES_PATH, FIGURES_PATH]:
    path.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Merged dataset not found. Run Notebook 01 first:\n"
        f"{DATA_FILE}"
    )

print(f"Dataset:       {DATA_FILE}")
print(f"Results path:  {RESULTS_PATH}")

## 4. Load, validate, and clean the data

In [ ]:
df = pd.read_csv(DATA_FILE)

required_columns = {
    TARGET,
    GROUP_COLUMN,
    *CONTINUOUS_FEATURES,
    *BINARY_FEATURES,
    *FACIAL_HAIR_FEATURES,
}

missing_columns = sorted(
    required_columns.difference(df.columns)
)

if missing_columns:
    raise KeyError(
        f"Required columns are missing: {missing_columns}"
    )

numeric_columns = [
    TARGET,
    *CONTINUOUS_FEATURES,
    *FACIAL_HAIR_FEATURES,
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce",
    )

In [ ]:
def convert_to_binary(series):
    mapping = {
        True: 1.0,
        False: 0.0,
        "true": 1.0,
        "false": 0.0,
        "True": 1.0,
        "False": 0.0,
        "yes": 1.0,
        "no": 0.0,
        "1": 1.0,
        "0": 0.0,
        1: 1.0,
        0: 0.0,
        1.0: 1.0,
        0.0: 0.0,
    }

    converted = series.map(mapping)
    invalid = series.notna() & converted.isna()

    if invalid.any():
        examples = (
            series.loc[invalid]
            .astype(str)
            .unique()[:5]
            .tolist()
        )

        raise ValueError(
            f"Unsupported binary values: {examples}"
        )

    return converted.astype(float)


for feature in BINARY_FEATURES:
    df[feature] = convert_to_binary(df[feature])

df = (
    df
    .dropna(subset=[TARGET, GROUP_COLUMN])
    .copy()
)

df[GROUP_COLUMN] = df[GROUP_COLUMN].astype(str)

available_groups = df[GROUP_COLUMN].dropna().unique().tolist()

group_order = [
    group
    for group in EXPECTED_GROUP_ORDER
    if group in available_groups
]

additional_groups = sorted(
    set(available_groups).difference(group_order)
)

group_order.extend(additional_groups)

if len(group_order) != 6:
    print(
        "Warning: expected six DiveFace groups but found "
        f"{len(group_order)}: {group_order}"
    )

print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")
display(
    df[GROUP_COLUMN]
    .value_counts()
    .reindex(group_order)
    .rename("n")
    .to_frame()
)

`age` does not appear in any feature list and is therefore excluded from all
analyses in this notebook.

## 5. Helper functions

In [ ]:
def apply_fdr(
    frame,
    p_column,
    output_column="p_value_fdr",
):
    result = frame.copy()
    result[output_column] = np.nan

    valid_mask = result[p_column].notna()

    if valid_mask.any():
        result.loc[valid_mask, output_column] = multipletests(
            result.loc[valid_mask, p_column],
            alpha=ALPHA,
            method=FDR_METHOD,
        )[1]

    return result


def safe_spearman(x, y):
    valid = pd.DataFrame(
        {"x": x, "y": y}
    ).dropna()

    if (
        len(valid) < 3
        or valid["x"].nunique() < 2
        or valid["y"].nunique() < 2
    ):
        return np.nan, np.nan, len(valid)

    rho, p_value = stats.spearmanr(
        valid["x"],
        valid["y"],
    )

    return float(rho), float(p_value), len(valid)


def cohens_d(group_0, group_1):
    group_0 = np.asarray(group_0, dtype=float)
    group_1 = np.asarray(group_1, dtype=float)

    if len(group_0) < 2 or len(group_1) < 2:
        return np.nan

    denominator = len(group_0) + len(group_1) - 2

    if denominator <= 0:
        return np.nan

    pooled_variance = (
        (len(group_0) - 1) * np.var(group_0, ddof=1)
        + (len(group_1) - 1) * np.var(group_1, ddof=1)
    ) / denominator

    if (
        not np.isfinite(pooled_variance)
        or pooled_variance <= 0
    ):
        return np.nan

    return float(
        (
            np.mean(group_1)
            - np.mean(group_0)
        )
        / np.sqrt(pooled_variance)
    )


def fit_with_robust_covariance(model, data):
    if (
        IDENTITY_COLUMN in data.columns
        and data[IDENTITY_COLUMN].nunique() > 1
    ):
        fitted_model = model.fit(
            cov_type="cluster",
            cov_kwds={
                "groups": data[IDENTITY_COLUMN]
            },
        )

        return fitted_model, "clustered_by_identity"

    return model.fit(cov_type="HC3"), "HC3"


def calculate_vif(frame):
    if frame.empty:
        return pd.Series(dtype=float)

    design = sm.add_constant(
        frame.astype(float),
        has_constant="add",
    )

    return pd.Series(
        [
            variance_inflation_factor(
                design.to_numpy(),
                feature_index,
            )
            for feature_index in range(
                1,
                design.shape[1],
            )
        ],
        index=frame.columns,
        name="vif",
        dtype=float,
    )

In [ ]:
def plot_heatmap(
    matrix,
    title,
    colorbar_label,
    filename,
    significance_matrix=None,
):
    if matrix.empty:
        print(f"No data available for: {title}")
        return

    values = matrix.to_numpy(dtype=float)
    finite_values = values[np.isfinite(values)]

    maximum_absolute_value = (
        max(
            np.nanmax(np.abs(finite_values)),
            1e-6,
        )
        if finite_values.size
        else 1.0
    )

    colormap = plt.get_cmap("coolwarm").copy()
    colormap.set_bad("lightgray")

    masked_values = np.ma.masked_invalid(values)

    figure, axis = plt.subplots(
        figsize=(
            max(9, 1.35 * matrix.shape[1]),
            max(4, 0.55 * matrix.shape[0]),
        )
    )

    image = axis.imshow(
        masked_values,
        aspect="auto",
        cmap=colormap,
        vmin=-maximum_absolute_value,
        vmax=maximum_absolute_value,
    )

    figure.colorbar(
        image,
        ax=axis,
        label=colorbar_label,
    )

    axis.set_xticks(range(matrix.shape[1]))
    axis.set_xticklabels(
        matrix.columns,
        rotation=30,
        ha="right",
    )

    axis.set_yticks(range(matrix.shape[0]))
    axis.set_yticklabels(matrix.index)
    axis.set_title(title)

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            value = values[row_index, column_index]

            if not np.isfinite(value):
                continue

            marker = ""

            if significance_matrix is not None:
                significance_value = significance_matrix.iloc[
                    row_index,
                    column_index,
                ]

                if (
                    pd.notna(significance_value)
                    and bool(significance_value)
                ):
                    marker = "*"

            axis.text(
                column_index,
                row_index,
                f"{value:.2f}{marker}",
                ha="center",
                va="center",
                fontsize=8,
            )

    plt.tight_layout()
    plt.savefig(
        FIGURES_PATH / filename,
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

Gray heatmap cells indicate that an effect could not be estimated because the
feature lacked sufficient variation or support in that group. An asterisk marks
a result that remains significant after FDR correction within the corresponding
analysis family.

## 6. Demographic CR-FIQA overview

In [ ]:
group_summary = (
    df.groupby(GROUP_COLUMN)[TARGET]
    .agg(
        n="count",
        mean="mean",
        median="median",
        std="std",
        minimum="min",
        maximum="max",
    )
    .reindex(group_order)
    .reset_index()
)

display(group_summary.round(4))

group_summary.to_csv(
    TABLES_PATH / "group_fiqa_summary.csv",
    index=False,
)

In [ ]:
group_values = [
    df.loc[
        df[GROUP_COLUMN] == group,
        TARGET,
    ]
    .dropna()
    .to_numpy()
    for group in group_order
]

plt.figure(figsize=(10, 5))
plt.boxplot(
    group_values,
    tick_labels=group_order,
    showfliers=False,
)
plt.xlabel("Demographic group")
plt.ylabel("CR-FIQA score")
plt.title("CR-FIQA Score Distribution by Demographic Group")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "fiqa_distribution_by_group.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

This overview is descriptive. The central research question of this notebook is
not whether average CR-FIQA scores differ, but whether **feature–score
relationships** differ across groups.

## 7. Group-wise univariate continuous effects

In [ ]:
continuous_rows = []

for group in group_order:
    group_df = df.loc[
        df[GROUP_COLUMN] == group
    ]

    for feature in CONTINUOUS_FEATURES:
        rho, p_value, sample_size = safe_spearman(
            group_df[feature],
            group_df[TARGET],
        )

        continuous_rows.append(
            {
                "group": group,
                "feature": feature,
                "n": sample_size,
                "spearman_rho": rho,
                "p_value": p_value,
            }
        )

continuous_results = apply_fdr(
    pd.DataFrame(continuous_rows),
    p_column="p_value",
    output_column="p_value_fdr",
)

continuous_results["significant_fdr"] = (
    continuous_results["p_value_fdr"] < ALPHA
)

display(continuous_results.round(4))

continuous_results.to_csv(
    TABLES_PATH
    / "continuous_groupwise_correlations.csv",
    index=False,
)

In [ ]:
continuous_matrix = (
    continuous_results
    .pivot(
        index="feature",
        columns="group",
        values="spearman_rho",
    )
    .reindex(
        index=CONTINUOUS_FEATURES,
        columns=group_order,
    )
)

continuous_significance = (
    continuous_results
    .pivot(
        index="feature",
        columns="group",
        values="significant_fdr",
    )
    .reindex(
        index=CONTINUOUS_FEATURES,
        columns=group_order,
    )
)

plot_heatmap(
    matrix=continuous_matrix,
    title=(
        "Spearman Correlations Between Continuous Features "
        "and CR-FIQA"
    ),
    colorbar_label="Spearman rho",
    filename="continuous_spearman_heatmap.png",
    significance_matrix=continuous_significance,
)

## 8. Group-wise univariate binary effects

In [ ]:
binary_rows = []

for group in group_order:
    group_df = df.loc[
        df[GROUP_COLUMN] == group
    ]

    for feature in BINARY_FEATURES:
        valid = group_df[
            [feature, TARGET]
        ].dropna()

        values_0 = valid.loc[
            valid[feature] == 0,
            TARGET,
        ].to_numpy()

        values_1 = valid.loc[
            valid[feature] == 1,
            TARGET,
        ].to_numpy()

        supported = (
            len(values_0) >= MIN_STATE_COUNT
            and len(values_1) >= MIN_STATE_COUNT
        )

        if supported:
            mann_whitney_result = stats.mannwhitneyu(
                values_0,
                values_1,
                alternative="two-sided",
            )

            p_value = float(
                mann_whitney_result.pvalue
            )

            mean_difference = float(
                np.mean(values_1)
                - np.mean(values_0)
            )

            effect_size = cohens_d(
                values_0,
                values_1,
            )
        else:
            p_value = np.nan
            mean_difference = np.nan
            effect_size = np.nan

        binary_rows.append(
            {
                "group": group,
                "feature": feature,
                "n_0": len(values_0),
                "n_1": len(values_1),
                "sufficient_support": supported,
                "mean_0": (
                    float(np.mean(values_0))
                    if len(values_0)
                    else np.nan
                ),
                "mean_1": (
                    float(np.mean(values_1))
                    if len(values_1)
                    else np.nan
                ),
                "mean_difference_1_minus_0": (
                    mean_difference
                ),
                "cohens_d": effect_size,
                "p_value": p_value,
            }
        )

binary_results = apply_fdr(
    pd.DataFrame(binary_rows),
    p_column="p_value",
    output_column="p_value_fdr",
)

binary_results["significant_fdr"] = (
    binary_results["p_value_fdr"] < ALPHA
)

display(binary_results.round(4))

binary_results.to_csv(
    TABLES_PATH / "binary_groupwise_effects.csv",
    index=False,
)

In [ ]:
binary_matrix = (
    binary_results
    .pivot(
        index="feature",
        columns="group",
        values="mean_difference_1_minus_0",
    )
    .reindex(
        index=BINARY_FEATURES,
        columns=group_order,
    )
)

binary_significance = (
    binary_results
    .pivot(
        index="feature",
        columns="group",
        values="significant_fdr",
    )
    .reindex(
        index=BINARY_FEATURES,
        columns=group_order,
    )
)

plot_heatmap(
    matrix=binary_matrix,
    title="Binary Feature Effects on CR-FIQA",
    colorbar_label=(
        "Mean difference: state 1 − state 0"
    ),
    filename="binary_effect_heatmap.png",
    significance_matrix=binary_significance,
)

A gray cell—such as a possible cell for `mask`—does not mean that the feature
has no effect. It means that the effect was not estimated because one binary
state did not meet the minimum support threshold in that demographic group.

## 9. Common adjusted regression specification

The same predictor set is used in all six group-specific models.

1. Continuous features are standardized once on the pooled sample.
2. Binary features are eligible only when both states have at least
   `MIN_STATE_COUNT` observations in every group.
3. Predictors with excessive pooled VIF are removed iteratively.
4. The final common specification is then used without group-specific feature
   selection.

In [ ]:
model_columns = [
    TARGET,
    GROUP_COLUMN,
    *CONTINUOUS_FEATURES,
    *BINARY_FEATURES,
]

if IDENTITY_COLUMN in df.columns:
    model_columns.append(IDENTITY_COLUMN)

regression_df = df[model_columns].copy()

standardization_rows = []
standardized_terms = []

for feature in CONTINUOUS_FEATURES:
    mean_value = regression_df[feature].mean()
    standard_deviation = regression_df[feature].std()
    term = f"z_{feature}"

    if (
        pd.notna(standard_deviation)
        and standard_deviation > 0
    ):
        regression_df[term] = (
            regression_df[feature] - mean_value
        ) / standard_deviation
    else:
        regression_df[term] = np.nan

    standardized_terms.append(term)

    standardization_rows.append(
        {
            "feature": feature,
            "mean": mean_value,
            "standard_deviation": standard_deviation,
            "term": term,
        }
    )

standardization_parameters = pd.DataFrame(
    standardization_rows
)

In [ ]:
binary_eligibility_rows = []

for feature in BINARY_FEATURES:
    group_support = []

    for group in group_order:
        values = regression_df.loc[
            regression_df[GROUP_COLUMN] == group,
            feature,
        ].dropna()

        counts = values.value_counts()

        group_support.append(
            counts.get(0.0, 0) >= MIN_STATE_COUNT
            and counts.get(1.0, 0) >= MIN_STATE_COUNT
        )

    binary_eligibility_rows.append(
        {
            "feature": feature,
            "included_before_vif": all(group_support),
            "supported_groups": int(
                np.sum(group_support)
            ),
            "required_groups": len(group_order),
        }
    )

binary_eligibility = pd.DataFrame(
    binary_eligibility_rows
)

eligible_binary_features = (
    binary_eligibility.loc[
        binary_eligibility["included_before_vif"],
        "feature",
    ]
    .tolist()
)

candidate_terms = (
    standardized_terms
    + eligible_binary_features
)

if not candidate_terms:
    raise ValueError(
        "No predictors have sufficient support for the common model."
    )

display(binary_eligibility)

In [ ]:
vif_complete_cases = (
    regression_df[candidate_terms]
    .dropna()
    .copy()
)

selected_terms = candidate_terms.copy()
removed_for_vif_rows = []

while len(selected_terms) > 1:
    vif_values = calculate_vif(
        vif_complete_cases[selected_terms]
    )

    finite_vif = vif_values.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    if (
        finite_vif.dropna().empty
        or finite_vif.max() <= MAX_VIF
    ):
        break

    term_to_drop = finite_vif.idxmax()

    removed_for_vif_rows.append(
        {
            "term": term_to_drop,
            "vif_at_removal": float(
                finite_vif.loc[term_to_drop]
            ),
        }
    )

    selected_terms.remove(term_to_drop)

removed_for_vif = pd.DataFrame(
    removed_for_vif_rows
)

final_vif = (
    calculate_vif(
        regression_df[selected_terms]
        .dropna()
    )
    .rename_axis("term")
    .reset_index(name="vif")
)

predictor_status = pd.DataFrame(
    {"term": candidate_terms}
)

predictor_status["feature"] = (
    predictor_status["term"]
    .str.replace("z_", "", regex=False)
)

predictor_status["feature_type"] = np.where(
    predictor_status["term"].str.startswith("z_"),
    "continuous",
    "binary",
)

predictor_status["included_in_common_model"] = (
    predictor_status["term"].isin(selected_terms)
)

predictor_status["status"] = np.where(
    predictor_status["included_in_common_model"],
    "included",
    "removed_high_vif",
)

display(predictor_status)
display(final_vif.round(3))

print("Final common predictors:")
print(selected_terms)

In [ ]:
standardization_parameters.to_csv(
    TABLES_PATH / "pooled_standardization_parameters.csv",
    index=False,
)

binary_eligibility.to_csv(
    TABLES_PATH / "binary_common_model_eligibility.csv",
    index=False,
)

predictor_status.to_csv(
    TABLES_PATH / "common_predictor_status.csv",
    index=False,
)

removed_for_vif.to_csv(
    TABLES_PATH / "predictors_removed_for_vif.csv",
    index=False,
)

final_vif.to_csv(
    TABLES_PATH / "final_common_predictor_vif.csv",
    index=False,
)

## 10. Group-specific adjusted regressions

In [ ]:
regression_rows = []
model_rows = []

for group in group_order:
    required_model_columns = [
        TARGET,
        *selected_terms,
    ]

    if IDENTITY_COLUMN in regression_df.columns:
        required_model_columns.append(
            IDENTITY_COLUMN
        )

    model_data = (
        regression_df.loc[
            regression_df[GROUP_COLUMN] == group,
            required_model_columns,
        ]
        .dropna()
        .copy()
    )

    X = sm.add_constant(
        model_data[selected_terms].astype(float),
        has_constant="add",
    )

    y = model_data[TARGET].astype(float)

    if len(model_data) <= X.shape[1]:
        raise ValueError(
            f"Too few complete observations for group: {group}"
        )

    if (
        np.linalg.matrix_rank(X.to_numpy())
        != X.shape[1]
    ):
        raise ValueError(
            f"Rank-deficient design matrix for group: {group}"
        )

    base_model = sm.OLS(y, X)

    fitted_model, covariance = fit_with_robust_covariance(
        base_model,
        model_data,
    )

    confidence_intervals = fitted_model.conf_int()

    model_rows.append(
        {
            "group": group,
            "n": int(fitted_model.nobs),
            "covariance": covariance,
            "r_squared": float(
                fitted_model.rsquared
            ),
            "adjusted_r_squared": float(
                fitted_model.rsquared_adj
            ),
        }
    )

    for term in selected_terms:
        regression_rows.append(
            {
                "group": group,
                "term": term,
                "feature": term.replace("z_", ""),
                "feature_type": (
                    "continuous"
                    if term.startswith("z_")
                    else "binary"
                ),
                "coefficient": float(
                    fitted_model.params[term]
                ),
                "std_error": float(
                    fitted_model.bse[term]
                ),
                "ci_lower": float(
                    confidence_intervals.loc[term, 0]
                ),
                "ci_upper": float(
                    confidence_intervals.loc[term, 1]
                ),
                "p_value": float(
                    fitted_model.pvalues[term]
                ),
            }
        )

group_models = pd.DataFrame(model_rows)

group_coefficients = apply_fdr(
    pd.DataFrame(regression_rows),
    p_column="p_value",
    output_column="p_value_fdr",
)

group_coefficients["significant_fdr"] = (
    group_coefficients["p_value_fdr"] < ALPHA
)

display(group_models.round(4))
display(group_coefficients.round(4))

In [ ]:
group_models.to_csv(
    TABLES_PATH / "group_regression_models.csv",
    index=False,
)

group_coefficients.to_csv(
    TABLES_PATH / "group_regression_coefficients.csv",
    index=False,
)

coefficient_matrix = (
    group_coefficients
    .pivot(
        index="feature",
        columns="group",
        values="coefficient",
    )
    .reindex(columns=group_order)
)

coefficient_significance = (
    group_coefficients
    .pivot(
        index="feature",
        columns="group",
        values="significant_fdr",
    )
    .reindex(columns=group_order)
)

plot_heatmap(
    matrix=coefficient_matrix,
    title=(
        "Adjusted Feature Coefficients by "
        "Demographic Group"
    ),
    colorbar_label="Regression coefficient",
    filename="adjusted_coefficient_heatmap.png",
    significance_matrix=coefficient_significance,
)

The coefficient heatmap uses the same specification in every group. This makes
cross-group comparisons more meaningful than fitting a separate feature
selection procedure within each demographic subset.

## 11. Formal feature × demographic-group interaction tests

In [ ]:
interaction_columns = [
    TARGET,
    GROUP_COLUMN,
    *selected_terms,
]

if IDENTITY_COLUMN in regression_df.columns:
    interaction_columns.append(
        IDENTITY_COLUMN
    )

interaction_data = (
    regression_df[interaction_columns]
    .dropna()
    .copy()
)

interaction_rows = []

for focal_term in selected_terms:
    control_terms = [
        term
        for term in selected_terms
        if term != focal_term
    ]

    formula = (
        f"{TARGET} ~ "
        f"{focal_term} * C({GROUP_COLUMN})"
    )

    if control_terms:
        formula += " + " + " + ".join(
            control_terms
        )

    base_model = smf.ols(
        formula=formula,
        data=interaction_data,
    )

    fitted_model, covariance = fit_with_robust_covariance(
        base_model,
        interaction_data,
    )

    interaction_term_names = [
        term
        for term in fitted_model.params.index
        if ":" in term and focal_term in term
    ]

    if interaction_term_names:
        parameter_names = list(
            fitted_model.params.index
        )

        restriction_matrix = np.zeros(
            (
                len(interaction_term_names),
                len(parameter_names),
            )
        )

        for row_index, term in enumerate(
            interaction_term_names
        ):
            restriction_matrix[
                row_index,
                parameter_names.index(term),
            ] = 1.0

        wald_result = fitted_model.wald_test(
            restriction_matrix,
            scalar=True,
        )

        interaction_p_value = float(
            wald_result.pvalue
        )

        wald_statistic = float(
            wald_result.statistic
        )
    else:
        interaction_p_value = np.nan
        wald_statistic = np.nan

    interaction_rows.append(
        {
            "term": focal_term,
            "feature": focal_term.replace("z_", ""),
            "feature_type": (
                "continuous"
                if focal_term.startswith("z_")
                else "binary"
            ),
            "n": int(fitted_model.nobs),
            "covariance": covariance,
            "wald_statistic": wald_statistic,
            "interaction_p": interaction_p_value,
        }
    )

interaction_results = apply_fdr(
    pd.DataFrame(interaction_rows),
    p_column="interaction_p",
    output_column="interaction_p_fdr",
)

interaction_results["interaction_significant_fdr"] = (
    interaction_results["interaction_p_fdr"] < ALPHA
)

interaction_results = (
    interaction_results
    .sort_values(
        "interaction_p_fdr",
        na_position="last",
    )
    .reset_index(drop=True)
)

display(interaction_results.round(4))

interaction_results.to_csv(
    TABLES_PATH / "feature_group_interactions.csv",
    index=False,
)

Each interaction p-value is a joint Wald test of whether the focal feature's
adjusted slope or binary effect differs anywhere across the demographic groups.
FDR correction is then applied across the tested features.

## 12. Facial-hair analysis in male groups

`moustache`, `beard`, and `sideburns` are graded features. They are analyzed
only in male groups and only when each male group contains sufficient zero,
non-zero, and unique values.

The adjusted models use the stable common-model predictors as controls and add
the supported standardized facial-hair features.

In [ ]:
male_groups = [
    group
    for group in group_order
    if "Man" in group
]

hair_support_rows = []

for group in male_groups:
    group_df = df.loc[
        df[GROUP_COLUMN] == group
    ]

    for feature in FACIAL_HAIR_FEATURES:
        values = group_df[feature].dropna()

        n_zero = int((values == 0).sum())
        n_nonzero = int((values > 0).sum())
        n_unique = int(values.nunique())

        hair_support_rows.append(
            {
                "group": group,
                "feature": feature,
                "n_zero": n_zero,
                "n_nonzero": n_nonzero,
                "n_unique": n_unique,
                "sufficient_support": (
                    n_zero >= MIN_STATE_COUNT
                    and n_nonzero >= MIN_NONZERO_HAIR
                    and n_unique >= 3
                ),
            }
        )

hair_support = pd.DataFrame(
    hair_support_rows
)

supported_hair_features = []

for feature in FACIAL_HAIR_FEATURES:
    feature_support = hair_support.loc[
        hair_support["feature"] == feature,
        "sufficient_support",
    ]

    if (
        len(feature_support) == len(male_groups)
        and feature_support.all()
    ):
        supported_hair_features.append(feature)

display(hair_support)

print(
    "Facial-hair features supported in all male groups:",
    supported_hair_features or "None",
)

In [ ]:
hair_results = pd.DataFrame()

if supported_hair_features:
    hair_df = regression_df.copy()

    for feature in supported_hair_features:
        hair_df[feature] = df.loc[
            hair_df.index,
            feature,
        ]

    male_mask = hair_df[GROUP_COLUMN].isin(
        male_groups
    )

    hair_standardization_rows = []

    for feature in supported_hair_features:
        mean_value = hair_df.loc[
            male_mask,
            feature,
        ].mean()

        standard_deviation = hair_df.loc[
            male_mask,
            feature,
        ].std()

        standardized_term = f"z_{feature}"

        if (
            pd.notna(standard_deviation)
            and standard_deviation > 0
        ):
            hair_df[standardized_term] = (
                hair_df[feature] - mean_value
            ) / standard_deviation
        else:
            hair_df[standardized_term] = np.nan

        hair_standardization_rows.append(
            {
                "feature": feature,
                "mean_male_pooled": mean_value,
                "standard_deviation_male_pooled": (
                    standard_deviation
                ),
                "term": standardized_term,
            }
        )

    hair_standardization = pd.DataFrame(
        hair_standardization_rows
    )

    hair_terms = [
        f"z_{feature}"
        for feature in supported_hair_features
    ]

    male_model_terms = selected_terms + hair_terms
    hair_rows = []

    for group in male_groups:
        required_columns = [
            TARGET,
            *male_model_terms,
        ]

        if IDENTITY_COLUMN in hair_df.columns:
            required_columns.append(
                IDENTITY_COLUMN
            )

        model_data = (
            hair_df.loc[
                hair_df[GROUP_COLUMN] == group,
                required_columns,
            ]
            .dropna()
            .copy()
        )

        X = sm.add_constant(
            model_data[male_model_terms].astype(float),
            has_constant="add",
        )

        y = model_data[TARGET].astype(float)

        if len(model_data) <= X.shape[1]:
            print(
                f"Skipping facial-hair model for {group}: "
                "too few complete observations."
            )
            continue

        if (
            np.linalg.matrix_rank(X.to_numpy())
            != X.shape[1]
        ):
            print(
                f"Skipping facial-hair model for {group}: "
                "rank-deficient design matrix."
            )
            continue

        base_model = sm.OLS(y, X)

        fitted_model, covariance = fit_with_robust_covariance(
            base_model,
            model_data,
        )

        confidence_intervals = fitted_model.conf_int()

        for feature in supported_hair_features:
            term = f"z_{feature}"

            hair_rows.append(
                {
                    "group": group,
                    "feature": feature,
                    "n": int(fitted_model.nobs),
                    "covariance": covariance,
                    "coefficient": float(
                        fitted_model.params[term]
                    ),
                    "std_error": float(
                        fitted_model.bse[term]
                    ),
                    "ci_lower": float(
                        confidence_intervals.loc[term, 0]
                    ),
                    "ci_upper": float(
                        confidence_intervals.loc[term, 1]
                    ),
                    "p_value": float(
                        fitted_model.pvalues[term]
                    ),
                }
            )

    if hair_rows:
        hair_results = apply_fdr(
            pd.DataFrame(hair_rows),
            p_column="p_value",
            output_column="p_value_fdr",
        )

        hair_results["significant_fdr"] = (
            hair_results["p_value_fdr"] < ALPHA
        )
else:
    hair_standardization = pd.DataFrame()

    print(
        "No facial-hair feature has sufficient support "
        "in every male demographic group."
    )

if not hair_results.empty:
    display(hair_results.round(4))
else:
    print(
        "No facial-hair regression results were produced."
    )

In [ ]:
hair_support.to_csv(
    TABLES_PATH / "facial_hair_support.csv",
    index=False,
)

hair_results.to_csv(
    TABLES_PATH / "facial_hair_male_regression.csv",
    index=False,
)

if not hair_standardization.empty:
    hair_standardization.to_csv(
        TABLES_PATH
        / "facial_hair_standardization_parameters.csv",
        index=False,
    )

Facial-hair coefficients are not included in the six-group interaction analysis.
They are reported as a separate male-group result because the necessary support
conditions are not comparable across all six intersectional groups.

## 13. Paper-ready final evidence table

In [ ]:
continuous_ranges = (
    continuous_results
    .groupby("feature")
    .agg(
        univariate_min=("spearman_rho", "min"),
        univariate_max=("spearman_rho", "max"),
        supported_groups=("group", "nunique"),
        univariate_significant_groups=(
            "significant_fdr",
            "sum",
        ),
    )
    .reset_index()
    .assign(feature_type="continuous")
)

binary_ranges = (
    binary_results
    .groupby("feature")
    .agg(
        univariate_min=(
            "mean_difference_1_minus_0",
            "min",
        ),
        univariate_max=(
            "mean_difference_1_minus_0",
            "max",
        ),
        supported_groups=(
            "sufficient_support",
            "sum",
        ),
        univariate_significant_groups=(
            "significant_fdr",
            "sum",
        ),
    )
    .reset_index()
    .assign(feature_type="binary")
)

univariate_ranges = pd.concat(
    [
        continuous_ranges,
        binary_ranges,
    ],
    ignore_index=True,
)

adjusted_ranges = (
    group_coefficients
    .groupby(
        ["feature", "feature_type"]
    )
    .agg(
        adjusted_min=("coefficient", "min"),
        adjusted_max=("coefficient", "max"),
        adjusted_significant_groups=(
            "significant_fdr",
            "sum",
        ),
    )
    .reset_index()
)

In [ ]:
final_summary = univariate_ranges.merge(
    predictor_status[
        [
            "feature",
            "feature_type",
            "included_in_common_model",
            "status",
        ]
    ],
    on=["feature", "feature_type"],
    how="left",
)

final_summary = final_summary.merge(
    adjusted_ranges,
    on=["feature", "feature_type"],
    how="left",
)

final_summary = final_summary.merge(
    interaction_results[
        [
            "feature",
            "feature_type",
            "interaction_p",
            "interaction_p_fdr",
            "interaction_significant_fdr",
        ]
    ],
    on=["feature", "feature_type"],
    how="left",
)


def interpret_final_result(row):
    included = row.get(
        "included_in_common_model",
        False,
    )

    interaction_significant = row.get(
        "interaction_significant_fdr",
        False,
    )

    if pd.isna(included) or included is not True:
        return (
            "not assessed in the common adjusted model"
        )

    if (
        pd.notna(interaction_significant)
        and interaction_significant is True
    ):
        return (
            "evidence of demographic heterogeneity"
        )

    return (
        "no FDR-significant demographic heterogeneity"
    )


final_summary["interpretation"] = (
    final_summary.apply(
        interpret_final_result,
        axis=1,
    )
)

In [ ]:
if not hair_results.empty:
    hair_summary = (
        hair_results
        .groupby("feature")
        .agg(
            supported_groups=("group", "nunique"),
            adjusted_min=("coefficient", "min"),
            adjusted_max=("coefficient", "max"),
            adjusted_significant_groups=(
                "significant_fdr",
                "sum",
            ),
        )
        .reset_index()
    )

    hair_summary["feature_type"] = "facial_hair"
    hair_summary["univariate_min"] = np.nan
    hair_summary["univariate_max"] = np.nan
    hair_summary["univariate_significant_groups"] = np.nan
    hair_summary["included_in_common_model"] = False
    hair_summary["status"] = "male_groups_only"
    hair_summary["interaction_p"] = np.nan
    hair_summary["interaction_p_fdr"] = np.nan
    hair_summary["interaction_significant_fdr"] = np.nan
    hair_summary["interpretation"] = (
        "separate adjusted evidence for male groups only"
    )

    final_summary = pd.concat(
        [
            final_summary,
            hair_summary,
        ],
        ignore_index=True,
    )

final_summary = (
    final_summary[
        [
            "feature",
            "feature_type",
            "supported_groups",
            "univariate_significant_groups",
            "included_in_common_model",
            "status",
            "univariate_min",
            "univariate_max",
            "adjusted_min",
            "adjusted_max",
            "adjusted_significant_groups",
            "interaction_p",
            "interaction_p_fdr",
            "interaction_significant_fdr",
            "interpretation",
        ]
    ]
    .sort_values(
        ["feature_type", "feature"]
    )
    .reset_index(drop=True)
)

display(final_summary.round(4))

final_summary.to_csv(
    TABLES_PATH
    / "final_demographic_consistency_summary.csv",
    index=False,
)

## 14. Automated conclusion

In [ ]:
heterogeneous_features = final_summary.loc[
    final_summary[
        "interaction_significant_fdr"
    ].eq(True),
    "feature",
].tolist()

stable_features = final_summary.loc[
    final_summary["interpretation"].eq(
        "no FDR-significant demographic heterogeneity"
    ),
    "feature",
].tolist()

not_assessed_features = final_summary.loc[
    final_summary["interpretation"].eq(
        "not assessed in the common adjusted model"
    ),
    "feature",
].tolist()

print("Paper-ready conclusion")
print("----------------------")

print(
    "Formal interaction tests are the main criterion "
    "for demographic heterogeneity."
)

print(
    "Features with FDR-significant interactions:",
    heterogeneous_features or "None",
)

print(
    "Features without FDR-significant interactions:",
    stable_features or "None",
)

print(
    "Features not assessed in the common adjusted model:",
    not_assessed_features or "None",
)

print(
    "Facial-hair findings are reported separately "
    "and are limited to male demographic groups."
)

## 15. Interpretation guidance

Use the following hierarchy when reporting the results:

1. **Formal interaction result**  
   This is the primary evidence for or against demographic heterogeneity.

2. **Adjusted group-specific coefficients**  
   These show where an interaction pattern may originate.

3. **Univariate group-wise effects**  
   These are descriptive supporting evidence.

4. **Support status**  
   Gray or missing cells mean that the effect could not be estimated reliably,
   not that the feature had no relationship with CR-FIQA.

A statistically significant effect in one group and a non-significant effect in
another group does not by itself prove that the effects differ. The interaction
test is required for that conclusion.

All results are observational associations and do not establish causality.

## 16. Save run metadata

In [ ]:
run_metadata = {
    "target": TARGET,
    "group_column": GROUP_COLUMN,
    "identity_column": (
        IDENTITY_COLUMN
        if IDENTITY_COLUMN in df.columns
        else None
    ),
    "group_order": group_order,
    "continuous_features": CONTINUOUS_FEATURES,
    "binary_features": BINARY_FEATURES,
    "facial_hair_features": FACIAL_HAIR_FEATURES,
    "age_included": False,
    "minimum_binary_state_count": MIN_STATE_COUNT,
    "minimum_nonzero_hair_count": MIN_NONZERO_HAIR,
    "maximum_vif": MAX_VIF,
    "alpha": ALPHA,
    "fdr_method": FDR_METHOD,
    "selected_common_terms": selected_terms,
    "supported_hair_features": supported_hair_features,
    "heterogeneous_features_fdr": heterogeneous_features,
    "stable_features_fdr": stable_features,
    "not_assessed_in_common_model": not_assessed_features,
}

with (
    RESULTS_PATH / "run_metadata.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        run_metadata,
        file,
        indent=2,
    )

print("Saved Notebook 08 run metadata.")

## 17. Saved-file summary

In [ ]:
print("Notebook 08 result files:")

for file_path in sorted(
    RESULTS_PATH.rglob("*")
):
    if file_path.is_file():
        print(
            "-",
            file_path.relative_to(
                RESULTS_PATH
            ),
        )

## 18. Main outputs for the paper

The most important outputs are:

- `tables/final_demographic_consistency_summary.csv`
- `tables/feature_group_interactions.csv`
- `tables/group_regression_coefficients.csv`
- `figures/adjusted_coefficient_heatmap.png`
- `figures/fiqa_distribution_by_group.png`

Together, these separate:

- overall demographic score differences;
- descriptive group-specific feature effects;
- adjusted group-specific effects;
- formal statistical evidence of demographic heterogeneity;
- features that could not be evaluated because of insufficient support.